# Tutorial 04 · Convolutional Neural Networks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JimengShi/SysEng_Cornell/blob/main/Tutorial_04_Convolutional_Neural_Networks/tutorial_04_cnn.ipynb)

Compare a dense network with a CNN on handwritten digits, then study regularization and data augmentation on small color images. Inspect what changes and what the evidence actually supports.

**Start:** Open in Colab → connect → **Runtime → Run all**, or use Shift+Enter to work cell by cell. Required package setup is included below. No local installation, API key, or Drive mount is needed.

**Prerequisites:** Python, arrays, and the training workflow in Tutorial 03.

**Resources:** CPU supported in quick mode; optional GPU. Downloads MNIST (~12 MB compressed) and CIFAR-10 (~163 MB). Plan 60–90 minutes for reading and exercises; training time depends on hardware.

Each notebook is independent: earlier lessons are helpful but their runtime state and files are not required. While the GitHub repository is private, authorize Colab to access it, or download this notebook and use **File → Upload notebook** in Colab.

## Learning goals

Trace `[batch, channels, height, width]` through convolution and pooling, compare parameter counts, train on separate training and validation splits, distinguish dropout from weight decay, and apply augmentation only to training images.

The notebook runs independently. `QUICK_RUN=True` uses subsets and three epochs; `False` uses larger splits and ten epochs. Quick mode reduces training work, not the size of dataset downloads.

## 1. Environment

This cell installs missing packages inside the current runtime. It keeps an existing PyTorch/torchvision pair together and does not downgrade Colab’s installed stack. If imports fail after other experiments, start a fresh runtime and run from the top.

In [ ]:
import importlib.util
import subprocess
import sys

packages = {'torch': 'torch', 'torchvision': 'torchvision', 'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'PIL': 'pillow'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if "torch" in missing or "torchvision" in missing:
    missing = sorted(set(missing + ["torch", "torchvision"]))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("Required packages are already installed.")

In [ ]:
import json, platform, random, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset, Subset
import torchvision
from torchvision import datasets, transforms, models
from PIL import Image
from IPython.display import display
%matplotlib inline

SEED = 42
QUICK_RUN = True
DEVICE_PREFERENCE = "auto"  # "cpu" forces CPU; "auto" uses CUDA when available.
assert DEVICE_PREFERENCE in {"auto", "cpu"}
device = torch.device("cuda" if DEVICE_PREFERENCE == "auto" and torch.cuda.is_available() else "cpu")
def seed_everything():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_everything()
torch.set_num_threads(min(2, torch.get_num_threads()))
DATA_DIR = Path("data") / "tutorial_04"
OUTPUT_DIR = Path("outputs") / "tutorial_04"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
versions = {"python": platform.python_version(), "torch": str(torch.__version__),
            "torchvision": str(torchvision.__version__), "numpy": np.__version__,
            "pandas": pd.__version__, "matplotlib": matplotlib.__version__,
            "device": str(device), "seed": SEED, "quick_run": QUICK_RUN}
display(pd.Series(versions, name="Environment"))
run_started = time.perf_counter()

## 2. What does a convolution do?

A convolution learns a small filter shared across spatial positions. For one spatial dimension, the output size is $\lfloor(H+2P-D(K-1)-1)/S+1\rfloor$, where $K$ is kernel size, $P$ padding, $S$ stride, and $D$ dilation.

PyTorch stores images as **NCHW**. With a 3 × 3 kernel, stride 1, and padding 1, a 28 × 28 image remains 28 × 28. A 2 × 2 max pool halves each spatial dimension. A convolution is approximately translation-equivariant away from boundaries; a full CNN is not automatically invariant to every translation.

The fixed filter below is a hand-written edge detector. Later, the CNN learns its own filters from data. `Conv2d` actually computes cross-correlation without flipping the kernel, the usual convention in deep learning.

In [ ]:
toy = torch.zeros(1, 1, 8, 8)
toy[:, :, :, 4:] = 1
edge_kernel = torch.tensor([[-1., 0., 1.], [-1., 0., 1.], [-1., 0., 1.]]).view(1, 1, 3, 3)
edge_response = F.conv2d(toy, edge_kernel, padding=1)
pooled = F.max_pool2d(edge_response, 2)
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, value, title in zip(axes, [toy, edge_response, pooled], ["Input", "Edge response", "Max pool"]):
    ax.imshow(value[0, 0], cmap="coolwarm")
    ax.set_title(f"{title}: {tuple(value.shape)}", fontsize=9)
    ax.axis("off")
plt.show()
assert edge_response.shape == (1, 1, 8, 8) and pooled.shape == (1, 1, 4, 4)

## 3. Prepare MNIST without test-set leakage

MNIST contains 28 × 28 grayscale digit images. Its official test split stays separate. We take validation images only from the official training split. Fixed pixel scaling maps `[0, 255]` to `[-1, 1]`; it fits no statistics to held-out images.

Quick split: **6,000 training / 1,000 validation / 2,000 test**. Full split: **54,000 / 6,000 / 10,000**. A fixed random permutation makes the choice reproducible. The download utility verifies dataset checksums.

In [ ]:
mnist_source = datasets.MNIST(str(DATA_DIR), train=True, download=True)
mnist_test_source = datasets.MNIST(str(DATA_DIR), train=False, download=True)

def split_indices(size, train_count, val_count, seed=SEED):
    order = np.random.default_rng(seed).permutation(size)
    train_idx, val_idx = order[:train_count], order[train_count:train_count+val_count]
    assert len(train_idx) == train_count and len(val_idx) == val_count
    assert not set(train_idx) & set(val_idx)
    return train_idx, val_idx

mnist_train_idx, mnist_val_idx = split_indices(len(mnist_source), 6000 if QUICK_RUN else 54000,
                                               1000 if QUICK_RUN else 6000)
mnist_test_idx = np.random.default_rng(SEED + 1).permutation(len(mnist_test_source))[:2000 if QUICK_RUN else 10000]

def mnist_subset(source, indices):
    return TensorDataset(source.data[indices].unsqueeze(1).float() / 127.5 - 1,
                         source.targets[indices].long())

mnist_train = mnist_subset(mnist_source, mnist_train_idx)
mnist_val = mnist_subset(mnist_source, mnist_val_idx)
mnist_test = mnist_subset(mnist_test_source, mnist_test_idx)
fig, axes = plt.subplots(2, 5, figsize=(8, 3))
for i, ax in enumerate(axes.flat):
    x, y = mnist_train[i]
    ax.imshow(x[0], cmap="gray", vmin=-1, vmax=1)
    ax.set_title(f"Digit {y.item()}")
    ax.axis("off")
plt.tight_layout()
plt.show()
print("MNIST split sizes:", len(mnist_train), len(mnist_val), len(mnist_test))

## 4. Dense network versus CNN

The dense model treats the image as 784 unrelated inputs. The CNN retains spatial structure through two convolution/pooling stages before a classifier. Both return ten logits for cross-entropy.

Convolution parameter count is `(kernel_height × kernel_width × input_channels + 1) × output_channels`, including bias. Pooling and flattening add no learned parameters. Fewer parameters are not a guarantee of higher accuracy; compare held-out results.

In [ ]:
class DenseDigits(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(nn.Flatten(), nn.Linear(784, 64), nn.ReLU(), nn.Linear(64, 10))
    def forward(self, x):
        return self.network(x)

class SmallCNN(nn.Module):
    def __init__(self, channels=1, side=28, dropout=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(channels, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(32*(side//4)**2, 64),
                                        nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 10))
    def forward(self, x):
        return self.classifier(self.features(x))

shape_model = SmallCNN()
x = torch.zeros(2, 1, 28, 28)
for layer in shape_model.features:
    x = layer(x)
    print(type(layer).__name__, tuple(x.shape))
assert x.shape == (2, 32, 7, 7)
print("First convolution parameters:", sum(p.numel() for p in shape_model.features[0].parameters()))
assert sum(p.numel() for p in shape_model.features[0].parameters()) == (3*3*1+1)*16
display(pd.Series({"Dense":sum(p.numel() for p in DenseDigits().parameters()),
                   "CNN":sum(p.numel() for p in SmallCNN().parameters())}, name="Parameter count"))

### Training helpers

The loop clears gradients, computes cross-entropy, backpropagates, and updates weights. Epoch losses are weighted by batch size. Validation runs without gradients and selects the lowest-loss checkpoint. Test data do not enter this loop.

We use AdamW; its decoupled weight decay shrinks weights separately from the gradient update. Dropout randomly masks activations during training and is disabled by `eval()`.

In [ ]:
def make_loader(dataset, shuffle=False, batch_size=128):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0,
                      generator=torch.Generator().manual_seed(SEED))

def one_epoch(model, loader, optimizer=None, augment=None):
    training = optimizer is not None
    model.train(training)
    loss_sum, correct, count = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if training and augment is not None:
                images = augment(images)
            logits = model(images)
            loss = F.cross_entropy(logits, labels)
            assert torch.isfinite(loss), "Non-finite loss"
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            loss_sum += loss.item() * len(labels)
            correct += (logits.argmax(1) == labels).sum().item()
            count += len(labels)
    return {"loss": loss_sum / count, "accuracy": correct / count}

def fit_classifier(model, training, validation, epochs, weight_decay=0.0, augment=None, lr=0.001):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    history, best_loss, best_state, best_epoch = [], float("inf"), None, 0
    for epoch in range(1, epochs + 1):
        train = one_epoch(model, training, optimizer, augment)
        val = one_epoch(model, validation)
        history.append({"epoch": epoch, "train_loss": train["loss"], "val_loss": val["loss"],
                        "train_accuracy": train["accuracy"], "val_accuracy": val["accuracy"]})
        if val["loss"] < best_loss:
            best_loss, best_epoch = val["loss"], epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"Epoch {epoch}: training={train['accuracy']:.1%}, validation={val['accuracy']:.1%}")
    model.load_state_dict(best_state)
    model.eval()
    print(f"Restored validation-selected epoch {best_epoch}.")
    return pd.DataFrame(history), best_epoch

def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    for prefix, label in [("train", "Training"), ("val", "Validation")]:
        axes[0].plot(history.epoch, history[prefix + "_loss"], label=label)
        axes[1].plot(history.epoch, history[prefix + "_accuracy"], label=label)
    axes[0].set(xlabel="Epoch", ylabel="Cross-entropy")
    axes[1].set(xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1))
    for ax in axes: ax.legend()
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

In [ ]:
digit_models, digit_history, digit_epochs = {}, {}, {}
for name, factory in [("Dense", DenseDigits), ("CNN", SmallCNN)]:
    seed_everything()
    model = factory().to(device)
    print("\nTraining", name)
    history, epoch = fit_classifier(model, make_loader(mnist_train, shuffle=True),
        make_loader(mnist_val), epochs=3 if QUICK_RUN else 10)
    digit_models[name], digit_history[name], digit_epochs[name] = model, history, epoch
    plot_history(history, f"MNIST: {name}")

digit_validation = {name:one_epoch(model, make_loader(mnist_val)) for name, model in digit_models.items()}
selected_digit = min(digit_validation, key=lambda name: digit_validation[name]["loss"])
digit_test = {name:one_epoch(model, make_loader(mnist_test)) for name, model in digit_models.items()}
display(pd.DataFrame(digit_test).T.rename_axis("Prespecified model"))
print("Model selected by validation loss:", selected_digit)

**Interpretation:** Compare validation curves and the prespecified test results. This comparison changes architecture and parameter count together, so it is not a controlled estimate of the isolated effect of convolution. Neither model is guaranteed to win on every seed.

## 5. CIFAR-10: regularization and augmentation

CIFAR-10 has 32 × 32 RGB images in ten classes. It is harder than MNIST. We compare three prespecified training configurations on the **same split and initial weights**:

1. Plain CNN.
2. CNN with dropout and weight decay.
3. The same regularized CNN plus random crop and horizontal flip on training images only.

This is a small teaching experiment. Short runs may underfit; regularization and augmentation are not guaranteed to improve the measured result. Combining dropout and weight decay in one configuration does not isolate their individual effects.

In [ ]:
cifar_source = datasets.CIFAR10(str(DATA_DIR), train=True, download=True)
cifar_test_source = datasets.CIFAR10(str(DATA_DIR), train=False, download=True)
cifar_train_idx, cifar_val_idx = split_indices(len(cifar_source), 4000 if QUICK_RUN else 45000,
                                               1000 if QUICK_RUN else 5000)
cifar_test_idx = np.random.default_rng(SEED + 1).permutation(len(cifar_test_source))[:2000 if QUICK_RUN else 10000]

def cifar_subset(source, indices):
    images = torch.from_numpy(source.data[indices].copy()).permute(0, 3, 1, 2).float() / 127.5 - 1
    labels = torch.tensor(np.asarray(source.targets)[indices], dtype=torch.long)
    return TensorDataset(images, labels)

cifar_train = cifar_subset(cifar_source, cifar_train_idx)
cifar_val = cifar_subset(cifar_source, cifar_val_idx)
cifar_test = cifar_subset(cifar_test_source, cifar_test_idx)
cifar_classes = cifar_source.classes
augmentation = transforms.Compose([transforms.RandomCrop(32, padding=4, fill=-1),
                                    transforms.RandomHorizontalFlip()])

class AugmentedDataset(Dataset):
    def __init__(self, base, transform):
        self.base, self.transform = base, transform
    def __len__(self):
        return len(self.base)
    def __getitem__(self, index):
        image, label = self.base[index]
        return self.transform(image), label  # Independent random transform for each image.

image, label = cifar_train[0]
fig, axes = plt.subplots(1, 5, figsize=(10, 2.5))
for i, ax in enumerate(axes):
    sample = image if i == 0 else augmentation(image)
    ax.imshow(((sample + 1)/2).permute(1, 2, 0).clamp(0, 1))
    ax.set_title("Original" if i == 0 else f"Augmented {i}", fontsize=9)
    ax.axis("off")
fig.suptitle(f"Training-only transforms: {cifar_classes[label.item()]}")
plt.tight_layout()
plt.show()
assert cifar_train.tensors[0].shape[1:] == (3, 32, 32)
print("CIFAR-10 split sizes:", len(cifar_train), len(cifar_val), len(cifar_test))

In [ ]:
cifar_models, cifar_histories, cifar_epochs = {}, {}, {}
configs = [("Plain", 0.0, 0.0, False), ("Regularized", 0.3, 0.001, False),
           ("Regularized + augmentation", 0.3, 0.001, True)]
for name, dropout, decay, use_augmentation in configs:
    seed_everything()
    model = SmallCNN(channels=3, side=32, dropout=dropout).to(device)
    training_data = AugmentedDataset(cifar_train, augmentation) if use_augmentation else cifar_train
    print("\nTraining", name)
    history, epoch = fit_classifier(model, make_loader(training_data, shuffle=True),
        make_loader(cifar_val), epochs=3 if QUICK_RUN else 10, weight_decay=decay)
    cifar_models[name], cifar_histories[name], cifar_epochs[name] = model, history, epoch
    plot_history(history, f"CIFAR-10: {name}")

cifar_validation = {name:one_epoch(model, make_loader(cifar_val)) for name, model in cifar_models.items()}
selected_cifar = min(cifar_validation, key=lambda name:cifar_validation[name]["loss"])
cifar_test_results = {name:one_epoch(model, make_loader(cifar_test)) for name, model in cifar_models.items()}
majority_label = torch.bincount(cifar_train.tensors[1]).argmax()
cifar_baseline = (cifar_test.tensors[1] == majority_label).float().mean().item()
display(pd.DataFrame(cifar_test_results).T)
print("Selected by validation loss:", selected_cifar)
print(f"Training-majority baseline test accuracy: {cifar_baseline:.1%}")

### Inspect mistakes and record the experiment

These are the first nine test images in the fixed ordering, not a selection of successes. A green title means a correct prediction. Compare mistakes across classes; do not adjust model settings based on this test inspection.

Training accuracy for augmented data is measured on randomly altered images with dropout active; validation uses clean images with dropout off. Their gap is not a pure measurement of overfitting.

In [ ]:
winner = cifar_models[selected_cifar]
winner.eval()
with torch.no_grad():
    predictions = winner(cifar_test.tensors[0][:9].to(device)).argmax(1).cpu()
fig, axes = plt.subplots(3, 3, figsize=(9, 8))
for i, ax in enumerate(axes.flat):
    image, label = cifar_test[i]
    predicted = predictions[i].item()
    ax.imshow(((image + 1)/2).permute(1, 2, 0))
    ax.set_title(f"True: {cifar_classes[label]}\nPred: {cifar_classes[predicted]}",
                 color="green" if predicted == label else "darkred", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

results = {"environment":versions,
    "mnist":{"splits":[len(mnist_train),len(mnist_val),len(mnist_test)], "validation":digit_validation,
             "test":digit_test, "selected":selected_digit, "best_epochs":digit_epochs},
    "cifar10":{"splits":[len(cifar_train),len(cifar_val),len(cifar_test)], "validation":cifar_validation,
               "test":cifar_test_results, "selected":selected_cifar, "baseline_accuracy":cifar_baseline,
               "best_epochs":cifar_epochs},
    "elapsed_seconds":round(time.perf_counter()-run_started,2)}
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(results,indent=2,allow_nan=False))
print("TUTORIAL_04_COMPLETE")
print(json.dumps(results,indent=2))

## 6. Exercises and extensions

1. Change padding from 1 to 0. Calculate the resulting feature-map sizes and fix the classifier input width.
2. Add an experiment with only dropout and one with only weight decay, keeping everything else fixed.
3. Increase the number of epochs and compare validation curves. Does a three-epoch result tell the same story?
4. Try `ColorJitter`, `RandomGrayscale`, or a small affine transformation on training images. Explain which transformations preserve the label and which may not.
5. Implement additive Gaussian noise as a callable transform, clamp to the expected input range, and visualize it before training.
6. Explore Mixup or CutMix. These mix labels as well as images, so the loss targets must change; do not copy an image-only augmentation into the existing class-index training loop.

**Concept checks:** Pooling has no trainable weights; convolution shares weights across positions; images use NCHW here; `eval()` disables dropout; test images are never randomly augmented in this experiment.

## Troubleshooting and next steps

- Run cells from the top after a runtime restart; a saved notebook does not preserve Python variables.
- If a data download fails, check connectivity and rerun that cell. If imports fail, use a fresh runtime.
- Download files from `outputs/tutorial_04/` using Colab’s Files sidebar before disconnecting.
- Change one experimental setting at a time and record what you changed.

## Attribution

Adapted and reworked for independent Colab use from [PEESE Tutorial 04](https://github.com/PEESEgroup/SysEn6888/tree/0b326ded95e415d7a60660879aaa737e3c7d79ea/Tutorial_04_Basics_of_Vision_CNN), Copyright (c) 2025 PEESE. The upstream MIT notice is retained in the repository `LICENSE`. This adaptation does not imply course endorsement.

Data: [MNIST](https://yann.lecun.com/exdb/mnist/) and [CIFAR-10 by Alex Krizhevsky](https://www.cs.toronto.edu/~kriz/cifar.html), downloaded through torchvision with checksums. Data are not redistributed in this repository. References: [PyTorch convolution documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) and [torchvision transforms](https://docs.pytorch.org/vision/stable/transforms.html).